# BIM RAG — Task 03 Full Pipeline (Structured Import + Vectorization)

Runs the complete reusable pipeline through one call:

```python
result = ifc_to_db(r"C:\path\to\model.ifc")
```

This imports IFC entities and relationships, enables pgvector, generates
deterministic entity/relationship documents, and embeds them with
`BAAI/bge-m3` into the unified `rag_documents` table. Safe and idempotent
to re-run — already-valid rows (matching source/text hash, template, and
embedding model) are skipped rather than re-embedded.

**Crash-recovery safeguards in effect (see `tasks/task03.md`):** CUDA batch
size capped at 8 (never 64), every document is trimmed to a real
tokenizer-measured budget before encoding, PyTorch/tokenizer threads are
capped, and each batch is committed independently so an interruption loses
at most one in-flight batch.

**Reuse for another IFC file:** change only the path passed to `ifc_to_db`.

In [1]:
import json
import time

from bim_rag.config import IFC_SOURCE_PATH
from bim_rag.pipeline_structured import ifc_to_db

No stream support: No module named 'lark'


## Run the full pipeline

The only required argument is the IFC file path. Database configuration is
loaded internally from `.env` and never displayed.

In [2]:
t0 = time.time()
report = ifc_to_db(str(IFC_SOURCE_PATH))
elapsed = time.time() - t0
print(f"\nElapsed: {elapsed:.1f}s")

[ifc_to_db] Source: IFC Schependomlaan incl planningsdata.ifc
[ifc_to_db] Computing fingerprint...
[ifc_to_db] SHA-256: 57fafa59f03b18c0...
[ifc_to_db] Scanning IFC model...


[ifc_to_db] Schema=IFC2X3  Total=843172  Entities=6989  Relationships=3473
[ifc_to_db] Connecting to database...
[ifc_to_db] DB connection OK.
[ifc_to_db] Creating/verifying schema tables...
[ifc_to_db] Importing 6989 entities...
[ifc_to_db] Existing source model id=1


[ifc_to_db] Entities new=0  updated=6989  failures=0
[ifc_to_db] Importing 3473 relationships...


[ifc_to_db] Relationships 3473/3473...
[ifc_to_db] Rels new=0  updated=3473  failures=0
[ifc_to_db] Members total=17668  resolved=17668  unresolved=0
[ifc_to_db] Structured import complete.


C:\Users\kdgki\anaconda3\envs\bim_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...

[Stage 2] Preparing entity_description docs for 6989 entities...


[Stage 2] Entities to (re)generate: 0  skipped (valid): 6989

[Stage 2] Entity docs: new=0 updated=0 skipped_valid=6989 truncated=0 failures=0


[Stage 2] Preparing relationship_description docs for 3473 relationships...


[Stage 2] Relationships to (re)generate: 0  skipped (valid): 3473

[Stage 2] Rel docs: new=0 updated=0 skipped_valid=3473 truncated=0 failures=0
[ifc_to_db] Vector phase complete.



Elapsed: 73.2s


## Full report

In [3]:
print(json.dumps(report, indent=2, default=str))

{
  "timestamp": "2026-07-11T23:37:56.134183+00:00",
  "source_model_id": 1,
  "file_fingerprint_prefix": "57fafa59f03b18c0...",
  "ifc_schema": "IFC2X3",
  "total_ifc_entity_count": 843172,
  "eligible_entity_count": 6989,
  "relationship_count_ifc": 3473,
  "relationship_class_counts": {
    "IfcRelAssignsToProcess": 73,
    "IfcRelSequence": 42,
    "IfcRelAssignsTasks": 125,
    "IfcRelAggregates": 4,
    "IfcRelDefinesByProperties": 3228,
    "IfcRelContainedInSpatialStructure": 1
  },
  "entity_class_counts": {
    "IfcTask": 125,
    "IfcScheduleTimeControl": 125,
    "IfcWorkPlan": 1,
    "IfcWorkSchedule": 1,
    "IfcBuilding": 1,
    "IfcBuildingStorey": 1,
    "IfcBuildingElementProxy": 86,
    "IfcPropertySet": 3228,
    "IfcSlab": 279,
    "IfcWindow": 259,
    "IfcDoor": 205,
    "IfcRailing": 90,
    "IfcBeam": 174,
    "IfcWallStandardCase": 232,
    "IfcCovering": 1214,
    "IfcColumn": 23,
    "IfcStair": 9,
    "IfcWall": 648,
    "IfcMember": 9,
    "IfcBuildingElem

## Structured counts

In [4]:
print(f"Source model id       : {report['source_model_id']}")
print(f"IFC schema            : {report['ifc_schema']}")
print(f"Entities              : {report['eligible_entity_count']}")
print(f"Relationships         : {report['relationship_count_ifc']}")
print(f"Members total         : {report['members_total']}")
print(f"Members resolved      : {report['members_resolved']}")
print(f"Members unresolved    : {report['members_unresolved']}")
print(
    f"Extraction failures   : {report['entity_extraction_failures']} entity, "
    f"{report['relationship_extraction_failures']} relationship"
)

Source model id       : 1
IFC schema            : IFC2X3
Entities              : 6989
Relationships         : 3473
Members total         : 17668
Members resolved      : 17668
Members unresolved    : 0
Extraction failures   : 0 entity, 0 relationship


## Vectorization counts, device, and crash-recovery configuration

In [5]:
print(f"pgvector enabled      : {report['pgvector_enabled']}")
print(f"Execution device      : {report['execution_device']}")
print(f"Embedding model       : {report['embedding_model']}")
print(f"Template version      : {report['template_version']}")
print(f"CUDA batch size       : {report['cuda_batch_size']}")
print(f"Thread limit          : {report['thread_limit']}")
print(f"Token limit           : {report['token_limit']}")
print()
print(
    f"Entity docs           : new={report['entity_docs_new']} "
    f"updated={report['entity_docs_updated']} "
    f"skipped_valid={report['entity_docs_skipped_valid']} "
    f"truncated={report['entity_docs_truncated']} "
    f"failures={report['entity_embed_failures']}"
)
print(
    f"Relationship docs     : new={report['rel_docs_new']} "
    f"updated={report['rel_docs_updated']} "
    f"skipped_valid={report['rel_docs_skipped_valid']} "
    f"truncated={report['rel_docs_truncated']} "
    f"failures={report['rel_embed_failures']}"
)
print(f"Total rag_documents   : {report['total_rag_docs']}")
print(f"Last attempted batch  : {report['last_attempted_batch']}")
print(f"Warnings              : {report['warning_count']}")

expected_total = report['eligible_entity_count'] + report['relationship_count_ifc']
validation_status = (
    "VALIDATED"
    if report['total_rag_docs'] == expected_total
    and report['entity_embed_failures'] == 0
    and report['rel_embed_failures'] == 0
    else "INCOMPLETE"
)
print(f"\nFinal validation status: {validation_status}")

pgvector enabled      : True
Execution device      : CUDA (NVIDIA GeForce RTX 5080 Laptop GPU)
Embedding model       : BAAI/bge-m3
Template version      : v001
CUDA batch size       : 8
Thread limit          : 4
Token limit           : 2000

Entity docs           : new=0 updated=0 skipped_valid=6989 truncated=0 failures=0
Relationship docs     : new=0 updated=0 skipped_valid=3473 truncated=0 failures=0
Total rag_documents   : 10462
Last attempted batch  : {}
Warnings              : 0

Final validation status: VALIDATED


## Example: source-scoped entity similarity search

Every query below is scoped by `source_model_id`, matching the returned
`report['source_model_id']`. `db_url` is loaded internally and never
printed.

In [6]:
from sqlalchemy import create_engine, text

from bim_rag.config import get_db_url

engine = create_engine(get_db_url())
source_model_id = report["source_model_id"]

with engine.connect() as conn:
    seed = conn.execute(
        text(
            "SELECT entity_id, document_text, embedding FROM rag_documents "
            "WHERE source_kind='entity' AND source_model_id=:sid "
            "ORDER BY entity_id LIMIT 1"
        ),
        {"sid": source_model_id},
    ).fetchone()
    print("Seed entity:", seed.document_text[:150])
    print()

    neighbors = conn.execute(
        text(
            "SELECT r.entity_id, e.global_id, e.ifc_class, "
            "r.embedding <=> :seed_vec AS distance "
            "FROM rag_documents r JOIN ifc_entities e ON r.entity_id = e.id "
            "WHERE r.source_kind='entity' AND r.source_model_id=:sid "
            "ORDER BY r.embedding <=> :seed_vec LIMIT 5"
        ),
        {"seed_vec": str(seed.embedding), "sid": source_model_id},
    ).fetchall()
    for row in neighbors:
        print(
            f"  entity_id={row.entity_id} class={row.ifc_class} "
            f"global_id={row.global_id} cosine_distance={row.distance:.4f}"
        )

Seed entity: This is an IfcTask entity. Its name is 'Dakpannen'. Its IFC GlobalId is 3Gl3OtTfjFLv$9p5FCB_K6. Description: 122.

  entity_id=1 class=IfcTask global_id=3Gl3OtTfjFLv$9p5FCB_K6 cosine_distance=0.0000
  entity_id=17 class=IfcTask global_id=0XfdnzIA52m9Wn76EjzK$B cosine_distance=0.0092
  entity_id=13 class=IfcTask global_id=29hHgFeU55xA$dyixszi8f cosine_distance=0.1140
  entity_id=9 class=IfcTask global_id=3b0bakrfHAFBBIqr9n_AYO cosine_distance=0.1153
  entity_id=27 class=IfcTask global_id=24duwkXR16gwcR82ZBlR4$ cosine_distance=0.1184


## Example: source-scoped relationship similarity search

In [7]:
with engine.connect() as conn:
    seed_r = conn.execute(
        text(
            "SELECT relationship_id, document_text, embedding FROM rag_documents "
            "WHERE source_kind='relationship' AND source_model_id=:sid "
            "ORDER BY relationship_id LIMIT 1"
        ),
        {"sid": source_model_id},
    ).fetchone()
    print("Seed relationship:", seed_r.document_text[:150])
    print()

    neighbors_r = conn.execute(
        text(
            "SELECT r.relationship_id, rel.global_id, rel.ifc_class, "
            "r.embedding <=> :seed_vec AS distance "
            "FROM rag_documents r JOIN ifc_relationships rel ON r.relationship_id = rel.id "
            "WHERE r.source_kind='relationship' AND r.source_model_id=:sid "
            "ORDER BY r.embedding <=> :seed_vec LIMIT 5"
        ),
        {"seed_vec": str(seed_r.embedding), "sid": source_model_id},
    ).fetchall()
    for row in neighbors_r:
        print(
            f"  relationship_id={row.relationship_id} class={row.ifc_class} "
            f"global_id={row.global_id} cosine_distance={row.distance:.4f}"
        )

Seed relationship: This is an IfcRelAssignsToProcess relationship. Its GlobalId is 1HldNQUeX9gfVrKeanTlHF. RelatingProcess: IfcTask (STEP: 842380, GlobalId: 22KGcOc5P95R

  relationship_id=1 class=IfcRelAssignsToProcess global_id=1HldNQUeX9gfVrKeanTlHF cosine_distance=0.0000
  relationship_id=2 class=IfcRelAssignsToProcess global_id=1NkEtUvnn8_PuTCwWUNV$V cosine_distance=0.0082
  relationship_id=101 class=IfcRelAssignsToProcess global_id=2TpUXwj9P6SgtyScC6O8Aj cosine_distance=0.0233
  relationship_id=93 class=IfcRelAssignsToProcess global_id=1Fjle3PI14$R6KfTx3A1LP cosine_distance=0.0269
  relationship_id=4 class=IfcRelAssignsToProcess global_id=26pp9XZLf7MupqVXPLWefg cosine_distance=0.0318


## Example: SQL + RAG fusion through canonical IDs

Combines a structured SQL filter with a vector search, joined only through
`ifc_entities.id` / `ifc_relationships.id` — never by name or parsed text.

In [8]:
with engine.connect() as conn:
    sql_ids = {
        r[0]
        for r in conn.execute(
            text(
                "SELECT id FROM ifc_entities WHERE ifc_class='IfcWall' AND source_model_id=:sid"
            ),
            {"sid": source_model_id},
        ).fetchall()
    }

    wall_seed = conn.execute(
        text(
            "SELECT e.id, r.embedding FROM ifc_entities e "
            "JOIN rag_documents r ON r.entity_id = e.id "
            "WHERE e.ifc_class='IfcWall' AND e.source_model_id=:sid LIMIT 1"
        ),
        {"sid": source_model_id},
    ).fetchone()

    vector_ids = {
        r[0]
        for r in conn.execute(
            text(
                "SELECT entity_id FROM rag_documents "
                "WHERE source_kind='entity' AND source_model_id=:sid "
                "ORDER BY embedding <=> :seed_vec LIMIT 50"
            ),
            {"seed_vec": str(wall_seed.embedding), "sid": source_model_id},
        ).fetchall()
    }

    print(f"SQL filter (IfcWall)           : {len(sql_ids)} ifc_entities.id values")
    print(f"Vector search (top 50 nearest)  : {len(vector_ids)} ifc_entities.id values")
    print(f"Intersection via canonical id   : {len(sql_ids & vector_ids)} entities")

SQL filter (IfcWall)           : 648 ifc_entities.id values
Vector search (top 50 nearest)  : 50 ifc_entities.id values
Intersection via canonical id   : 17 entities


## Multi-IFC reuse

To process another IFC file, change only the path:

```python
result = ifc_to_db(r"C:\path\to\another_model.ifc")
```

A distinct fingerprint creates a distinct `source_model_id`; every
structured row, RAG document, and vector search above stays scoped to that
id, so results never mix across source models.